In [1]:
import pandas as pd
import sys
import time

sys.path.insert(0,'./chromedriver-win64/chromedriver-win64/chromedriver.exe')
# pip install webdriver-manager
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
import os
from selenium.webdriver.common.keys import Keys

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Definimos una espera máxima de 10 segundos



In [2]:
def click_function(using, element):
    return wait.until(EC.element_to_be_clickable((using, element)))

In [3]:
print(f"PATH: {os.listdir("./chromedriver-win64/chromedriver-win64")}")
service = Service("./chromedriver-win64/chromedriver-win64/chromedriver.exe")
#service = Service("chromedriver.exe")
options = webdriver.ChromeOptions()
options.add_argument("--incognito")

driver = webdriver.Chrome(service=service, options=options)
wait = WebDriverWait(driver, 25)
url = "https://www.imdb.com/es-es/"          
driver.get(url)

click_function(By.XPATH, '//*[@id="__next"]/div[1]/div/div[2]/div/button[1]').click()


PATH: ['.ipynb_checkpoints', 'chromedriver.exe', 'LICENSE.chromedriver', 'THIRD_PARTY_NOTICES.chromedriver']


In [17]:
click_function(By.XPATH, '//*[@id="nav-search-form"]/div[1]/div/span[1]').click()
click_function(By.XPATH, '//*[@id="nav-search-form"]/div[1]/div/div/div/div/ul/a').click()


In [12]:
click_function(By.XPATH, "//span[text()='Desplegar todo'] | //span[text()='Plegar todo']").click()

click_function(By.CSS_SELECTOR, 'button[data-testid="test-chip-id-movie"]').click()

click_function(By.CSS_SELECTOR, '[data-testid="releaseYearMonth-end"]').send_keys("2010")

boton = driver.find_element(By.CSS_SELECTOR, "button[data-testid='test-chip-id-Comedy']")
driver.execute_script("arguments[0].click();", boton)


click_function(By.XPATH, '//*[@id="__next"]/main/div[2]/div[3]/section/section/div/section/section/div[2]/div/section/div[1]/button').click()


In [13]:
link =  click_function(By.XPATH, '//*[@id="__next"]/main/div[2]/div[3]/section/section/div/section/section/div[2]/div/section/div[2]/div[2]/ul/li[1]/div/div/div/div[1]/div[2]/div[1]/a')


# 1. Obtener la URL del elemento
url_pelicula = link.get_attribute("href")

# 2. Guardar la pestaña actual
pestana_principal = driver.current_window_handle

# 3. Usar JavaScript para abrir la pestaña (esto hereda el modo incógnito)
driver.execute_script(f'window.open("{url_pelicula}", "_blank");')

# 4. Cambiar el foco a la nueva pestaña (la última en abrirse)
time.sleep(2) # Espera breve para que el navegador reaccione
driver.switch_to.window(driver.window_handles[-1])

# --- extraer datos aquí ---

# 5. Cerrar y volver


In [42]:

list = []


año = driver.find_element(By.XPATH, '//*[@id="__next"]/main/div/section[1]/section/div[3]/section/section/div[2]/div[1]/ul/li[1]/a').text
duracion = driver.find_element(By.XPATH, '//*[@id="__next"]/main/div/section[1]/section/div[3]/section/section/div[2]/div[1]/ul/li[3]').text
calificacion =  driver.find_element(By.XPATH, '//*[@id="__next"]/main/div/section[1]/div/section/div/div[1]/section[5]/div[2]/div[1]/span[1]/span').text     
director = driver.find_element(By.XPATH, '//*[@id="__next"]/main/div/section[1]/div/section/div/div[1]/section[4]/ul/li[1]/div/ul/li/a').text

reparto = "?"
sinopsis = driver.find_element(By.XPATH, '//*[@id="__next"]/main/div/section[1]/section/div[3]/section/section/div[3]/div[2]/div[1]/section/p/span[3]/span/span').text

pais = driver.find_element(By.XPATH, '//*[@id="__next"]/main/div/section[1]/div/section/div/div[1]/section[11]/div[2]/ul/li[2]/div/ul/li/a').text
pais = driver.find_element(By.XPATH, '//*[@id="__next"]/main/div/section[1]/div/section/div/div[1]/section[11]/div[2]/ul/li[2]/div/ul/li/a').text


In [52]:
actors_elements = driver.find_elements(By.XPATH, '//*[@id="__next"]/main/div/section[1]/section/div[3]/section/section/div[3]/div[2]/div[2]/div[2]/ul/li[3]/div/ul//li/a')
actores = []
for actor in actors_elements:
    actores.append(actor.text)

    

In [60]:
from bs4 import BeautifulSoup


html = driver.page_source

# 4. Procesar con BeautifulSoup
soup = BeautifulSoup(html, "html.parser")
print(soup)
# 5. Extraer datos (ejemplo: títulos de reviews)
titulos = [t.get_text(strip=True) for t in soup.select(".review-container .title")]

print(titulos)

<html class="" lang="es-ES" xmlns:fb="http://www.facebook.com/2008/fbml" xmlns:og="http://opengraphprotocol.org/schema/"><head><script async="" src="https://sb.scorecardresearch.com/beacon.js"></script><script async="" crossorigin="anonymous" src="https://images-na.ssl-images-amazon.com/images/I/21auWwkMYgL.js"></script><meta charset="utf-8"/><meta content="width=device-width" name="viewport"/><script>if(typeof uet === 'function'){ uet('bb', 'LoadTitle', {wb: 1}); }</script><title>Scary Movie (2000) - Reseñas de usuarios - IMDb</title><meta content="Scary Movie (2000) - Películas, televisión, celebridades y más..." data-id="main" name="description"/><meta content="Reseñas, horarios, DVD, fotos, foros, calificaciones, sinopsis, tráileres, créditos" name="keywords"/><meta content="0cadf7898134e79b" name="google-site-verification"/><meta content="C1DACEF2769068C0B0D2687C9E5105FA" name="msvalidate.01"/><meta content="max-image-preview:large" name="robots"/><meta content="https://www.imdb.c

### Funcion para recolectar


In [4]:
def recolectar_peliculas(genero, filter_year, id_pelicula):
    from selenium.webdriver.common.by import By

    click_function(By.XPATH, '//*[@id="nav-search-form"]/div[1]/div/span[1]').click()
    click_function(By.XPATH, '//*[@id="nav-search-form"]/div[1]/div/div/div/div/ul/a').click()

    click_function(By.XPATH, "//span[text()='Desplegar todo'] | //span[text()='Plegar todo']").click()

    click_function(By.CSS_SELECTOR, 'button[data-testid="test-chip-id-movie"]').click()

    a = f'[data-testid="{filter_year}"]'
    
    # click_function(By.CSS_SELECTOR, '[data-testid="releaseYearMonth-end"]').send_keys(filter_year[1])
    fl_1 = f'[data-testid="{filter_year[0]}"]'
    click_function(By.CSS_SELECTOR, fl_1).send_keys(filter_year[1])

    tipo = f"button[data-testid='test-chip-id-{genero}']"
    boton = click_function(By.CSS_SELECTOR, tipo)
    driver.execute_script("arguments[0].click();", boton)
    
    
    click_function(By.XPATH, '//*[@id="__next"]/main/div[2]/div[3]/section/section/div/section/section/div[2]/div/section/div[1]/button').click()

    path_pelicula = f'//*[@id="__next"]/main/div[2]/div[3]/section/section/div/section/section/div[2]/div/section/div[2]/div[2]/ul/li[{id_pelicula}]/div/div/div/div[1]/div[2]/ul/div/a'
    link =  click_function(By.XPATH, path_pelicula)

    # 1. Obtener la URL del elemento
    url_pelicula = link.get_attribute("href")
    
    # 2. Guardar la pestaña actual
    pestana_principal = driver.current_window_handle
    
    # 3. Usar JavaScript para abrir la pestaña (esto hereda el modo incógnito)
    driver.execute_script(f'window.open("{url_pelicula}", "_blank");')
    
    # 4. Cambiar el foco a la nueva pestaña (la última en abrirse)
    time.sleep(2) # Espera breve para que el navegador reaccione
    driver.switch_to.window(driver.window_handles[-1])
    
    # --- extraer datos aquí ---
    
    # 5. Cerrar y volver

    
    list = []
    
    titulo = click_function(By.XPATH, '//*[@id="__next"]/main/div/section[1]/section/div[3]/section/section/div[2]/div[1]/h1').text
    # año = click_function(By.XPATH, '//*[@id="__next"]/main/div/section[1]/section/div[3]/section/section/div[2]/div[1]/ul/li[1]/a').text
    # año = click_function(By.XPATH, '//*[@id="__next"]/main/div/section[1]/section/div[3]/section/section/div[2]/div[1]/ul/li[1]/a').text
    año = click_function(By.XPATH, "//li[@role='presentation']/a[contains(@href, 'releaseinfo')]").text
    duracion = click_function(By.XPATH, '//*[@id="__next"]/main/div/section[1]/section/div[3]/section/section/div[2]/div[1]/ul/li[3]').text
    nota = driver.find_element(By.CSS_SELECTOR, 'div[data-testid$="score"] span').text
    try: 
        calificacion =  click_function(By.XPATH, '//*[@id="__next"]/main/div/section[1]/div/section/div/div[1]/section[5]/div[2]/div[1]/span[1]/span').text     
        director = click_function(By.XPATH, '//*[@id="__next"]/main/div/section[1]/div/section/div/div[1]/section[4]/ul/li[1]/div/ul/li/a').text
    except  Exception as e:
        calificacion =  click_function(By.XPATH, '//*[@id="__next"]/main/div/section[1]/section/div[3]/section/section/div[3]/div[2]/div[2]/div[1]/div/div[1]/a/span/div/div[2]/div[1]/span[1]').text  
        director = click_function(By.XPATH, '//*[@id="__next"]/main/div/section[1]/section/div[3]/section/section/div[3]/div[2]/div[2]/div[2]/ul/li[1]/div/ul/li/a').text
    
    # sinopsis = click_function(By.XPATH, '//*[@id="__next"]/main/div/section[1]/section/div[3]/section/section/div[3]/div[2]/div[1]/section/p/span[3]/span/span').text  
    sinopsis = click_function(By.XPATH, '//*[@id="__next"]/main/div/section[1]/section/div[3]/section/section/div[3]/div[2]/div[1]/section/p').text
    # pais = click_function(By.XPATH, '//*[@id="__next"]/main/div/section[1]/div/section/div/div[1]/section[11]/div[2]/ul/li[2]/div/ul/li/a').text
    
    # año = driver.find_element(By.XPATH, '//*[@id="__next"]/main/div/section[1]/section/div[3]/section/section/div[2]/div[1]/ul/li[1]/a').text
    # duracion = driver.find_element(By.XPATH, '//*[@id="__next"]/main/div/section[1]/section/div[3]/section/section/div[2]/div[1]/ul/li[3]').text
    # calificacion =  driver.find_element(By.XPATH, '//*[@id="__next"]/main/div/section[1]/div/section/div/div[1]/section[5]/div[2]/div[1]/span[1]/span').text     
    # director = driver.find_element(By.XPATH, '//*[@id="__next"]/main/div/section[1]/div/section/div/div[1]/section[4]/ul/li[1]/div/ul/li/a').text
    
    # reparto = "?"
    # sinopsis = driver.find_element(By.XPATH, '//*[@id="__next"]/main/div/section[1]/section/div[3]/section/section/div[3]/div[2]/div[1]/section/p/span[3]/span/span').text
    
    # pais = driver.find_element(By.XPATH, '//*[@id="__next"]/main/div/section[1]/div/section/div/div[1]/section[11]/div[2]/ul/li[2]/div/ul/li/a').text
    # pais = driver.find_element(By.XPATH, '//*[@id="__next"]/main/div/section[1]/div/section/div/div[1]/section[11]/div[2]/ul/li[2]/div/ul/li/a').text
    actors_elements = driver.find_elements(By.XPATH, '//*[@id="__next"]/main/div/section[1]/section/div[3]/section/section/div[3]/div[2]/div[2]/div[2]/ul/li[3]/div/ul//li/a')
    actores = []
    time.sleep(6)
    for actor in actors_elements:
        actores.append(actor.text)
        
    # actors_elements = click_function(By.XPATH, '//*[@id="__next"]/main/div/section[1]/section/div[3]/section/section/div[3]/div[2]/div[2]/div[2]/ul/li[3]/div/ul//li/a')
    # actores = []
    # for actor in actors_elements:
    #     actores.append(actor.text)
        

    boton = click_function(By.XPATH, "//span[text()='Reseñas de usuarios']")
    driver.execute_script("arguments[0].click();", boton)

    
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    
    
    # Esperar a que el select exista en el DOM
    select_el = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.ID, "sort-by-selector"))
    )
    
    # Cambiar el valor con JS
    driver.execute_script("""
        const sel = arguments[0];
        sel.value = "SUBMISSION_DATE"; 
        sel.dispatchEvent(new Event('change', { bubbles: true }));
    """, select_el)

    reseña=[]
    for i in range(1,6):
        element = f'//*[@id="__next"]/main/div/section/div/section/div/div[1]/section[1]/article[{i}]/div[1]/div[1]'
        reseña_c = click_function(By.XPATH, element).text
        calif = reseña_c.split("\n")

        # print(calif)
        # califf = -1
        # if len(calif)>3:
        #     califf = calif[3]
            
        # if calif[3].lower()=='spoiler':
        #     califf=calif[2]
        reseña.append(calif)
        
    print(f"Título: {titulo}")
    print(f"Año: {año}")
    print(f"Duración: {duracion}")
    print(f"Calificación: {calificacion}")
    print(f"Director: {director}")
    print(f"Sinopsis: {sinopsis}")
    # print(f"País: {pais}")
    print(f"Actores: {actores}")
    print(f"Reseñas: {reseña}")
    
    info_pelis = {
        "Titulo": titulo,
        "Género": genero,
        "Año": año,
        "Duración": duracion,
        "Calificación": calificacion,
        "Director": director,
        "Protagonistas": actores,
        "Sinopsis": sinopsis,
        "Reseñas": reseña
    }
    time.sleep(1)
    driver.close()
    driver.switch_to.window(pestana_principal)
    return info_pelis

In [9]:
filter_year = ["releaseYearMonth-end", 2025]
info_pelis = recolectar_peliculas("Comedy", filter_year,1)

Título: Bugonia
Año: 2025
Duración: 1h 58min
Calificación: 7,4
Director: Yorgos Lanthimos
Sinopsis: Dos jovenes conspiranoicos secuestran a la CEO de una empresa multimillonaria, convencidos de que es una extraterrestre malvada que planea acabar con la raza humana.
País: Irlanda
Actores: ['Emma Stone', 'Jesse Plemons', 'Aidan Delbis']
Reseñas: [{'Calificación': '6', 'Reseña': "It was chugging along fine, till it turned up the kitch. Some movies pretend to be smart - this is one of them. Plenty of shock value, and good performances, but at the end of it, not much to take away really. Though it tries to be a comedy, a drama, a sci-fi all at once, it's none of these. Don't take my word for it - just see how it ages!"}, {'Calificación': '7', 'Reseña': "Before Bugonia, I'd only seen two Yorgos Lanthimos film: Poor Things, which I loved, and Kinds of Kindness, which I absolutely hated."}, {'Calificación': '1', 'Reseña': "Man Look, for content, there was no content, just absurdity and honestl

In [104]:
print(f"PATH: {os.listdir('../chromedriver-win64')}")
service = Service("../chromedriver-win64/chromedriver.exe")
#service = Service("chromedriver.exe")
options = webdriver.ChromeOptions()
options.add_argument("--incognito")

driver = webdriver.Chrome(service=service, options=options)
wait = WebDriverWait(driver, 25)
url = "https://www.imdb.com/es-es/"          
driver.get(url)

click_function(By.XPATH, '//*[@id="__next"]/div[1]/div/div[2]/div/button[1]').click()


PATH: ['chromedriver.exe', 'LICENSE.chromedriver', 'THIRD_PARTY_NOTICES.chromedriver']


In [105]:
lista_pel = []
gen = ["Action", "Comedy", "Drama", "Horror", "Animation"]

for g in gen:
    print("=======================================================================")
    print("=======================================================================")
    print("=======================================================================")
    print(g)
    try:
        for i in range(1,12):
            print(f'<--------{i}-------->')
            filter_year = ["releaseYearMonth-end", 2024]
            info_pelis = recolectar_peliculas(g, filter_year, i)
            lista_pel.append(info_pelis)
    except Exception as e:
        time.sleep(1)
        driver.close()
        driver.switch_to.window(pestana_principal)
    time.sleep(3)

Action
<--------1-------->
Título: Noche de bodas
Año: 2019
Duración: 1h 35min
Calificación: 6,9
Director: Matt Bettinelli-Olpin
Sinopsis: La noche de bodas de una novia da un giro siniestro cuando sus nuevos suegros la obligan a formar parte de un juego aterrador.
Actores: ['', '', '']
Reseñas: [['6', '/10', 'A decently entertaining dark comedy horror', "Ready or Not is a perfectly fine film with a fun concept, I just don't think it takes it far enough!", '', 'I had a lot of fun with the set up and the characters, but felt like there were a few missed opportunities. At times the story also seemed to lag a little bit and surrendered to cliche on a few too many occasions.', '', 'Samara Weaving was a perfect scream queen and an easy character to root for and get on board with.', '', 'Stylistically I liked the film, with its balance of horror, gore, and dark comedy. Again, it just could have been pushed a little further.', '', "Ultimately though it is a fun watch and it's got me looking f

In [106]:
import pandas as pd


df = pd.DataFrame(lista_pel)

df.to_csv('peliculas_imdb.csv', index=False, encoding='utf-16', sep=';')



In [ ]:
pritn(df[1])


In [93]:
filter_year = ["releaseYearMonth-end", 2024]
info_pelis = recolectar_peliculas("Action", filter_year, 10)

Título: Relay
Año: 2024
Duración: 1h 52min
Calificación: 7,0
Director: David Mackenzie
Sinopsis: Un intermediario de lucrativos sobornos entre empresas corruptas y los individuos que las amenazan rompe sus propias reglas cuando un nuevo cliente busca su protección para seguir con vida.
Actores: ['Riz Ahmed', 'Lily James', 'Sam Worthington']
Reseñas: [['9', '/10', 'DO NOT MISS!!!', 'This is what l call a great movie!', '', 'Unfortunately today we see a " 84% " then we watch movie and its 24%!', '', "Relay may be best movie l've watched this year Its got it all... & merits 95 - 100% hands down!", '', 'Anyone who likes a thriller should not miss this film!', '', "Riz Ahmed, is one of the best actors around today. Academy Award Winner... l'm positive he will be doing some grrat work in future!"], ['8', '/10', 'A tight, tense, well paced thriller...with a somewhat pedestrian final Act.', 'Like many of the reviews here, I found Relay to be a very compelling, well constructed, and interesting

In [110]:
df

,Titulo,Género,Año,Duración,Calificación,Director,Protagonistas,Sinopsis,Reseñas
0,Noche de bodas,Action,2019,1h 35min,"6,9",Matt Bettinelli-Olpin,"[, , ]",La noche de bodas de una novia da un giro sini...,"[[6, /10, A decently entertaining dark comedy ..."
1,El especialista,Action,2024,2h 6min,"6,8",David Leitch,"[Ryan Gosling, Emily Blunt, Aaron Taylor-Johnson]",Un doble de Hollywood trabaja como cazarrecomp...,"[[4, /10, disappointing, I like Ryan Gosling a..."
2,Sicario,Action,2015,2h 1min,"7,7",Denis Villeneuve,"[Emily Blunt, Josh Brolin, Benicio Del Toro]",Una idealista agente del FBI es reclutada por ...,"[[10, /10, A Masterpiece!, Sicario is a comple..."
3,El caballero oscuro,Action,2008,2h 32min,"9,1",Christopher Nolan,"[Christian Bale, Heath Ledger, Aaron Eckhart]",Cuando la amenaza conocida como el Joker causa...,"[[9, /10, Legendary Film, Heath Ledger deliver..."
4,Origen,Action,2010,2h 28min,"8,8",Christopher Nolan,"[Leonardo DiCaprio, Joseph Gordon-Levitt, Elli...",A un ladrón que roba secretos corporativos a t...,"[[10, /10, Inception, Spoiler], [10, /10, I ev..."
5,Heat,Action,1995,2h 50min,"8,3",Michael Mann,"[Al Pacino, Robert De Niro, Val Kilmer]",Un grupo de ladrones profesionales de alto niv...,"[[8, /10, Heat (1995), Spoiler], [9, /10, A Sy..."
6,Abigail,Action,2024,1h 49min,"6,5",Matt Bettinelli-Olpin,"[Melissa Barrera, Dan Stevens, Alisha Weir]",Después de que un grupo de criminales secuestr...,"[[5, /10, Started Strong with Gangster Vibes, ..."
7,El ministerio de la guerra sucia,Action,2024,2h 2min,"6,8",Guy Ritchie,"[Henry Cavill, Alan Ritchson, Alex Pettyfer]",El ejército británico recluta a un pequeño gru...,"[[10, /10, Stylish, Wild, and Surprisingly Fun..."
8,Jurassic Park (Parque Jurásico),Action,1993,2h 7min,"8,2",Steven Spielberg,"[Sam Neill, Laura Dern, Jeff Goldblum]","Gracias al ADN fosilizado en ámbar, John Hammo...","[[9, /10, Amazing work for sure, Few films hav..."
9,Relay,Action,2024,1h 52min,"7,0",David Mackenzie,"[Riz Ahmed, Lily James, Sam Worthington]",Un intermediario de lucrativos sobornos entre ...,"[[9, /10, DO NOT MISS!!!, This is what l call ..."


In [134]:
df['Reseñas'][0][3]

['8',
 '/10',
 'Good Premise, Great Acting, Entertaining Movie',
 'An entertaining movie with a great idea. It is action packed, tense and overall fun to watch.',
 '',
 'The soundtrack of the game featured in the movie fits very well and gives a good introduction to the overall feeling of it. The acting of Samara Weaving is spectacular and carries much of the film, although the overall setting is also perfectly chosen to allow for some social commentary.',
 '',
 'Some of the side characters lack depth and feel too exaggerated for their own good. Because of that, it can sometimes be a little boring to follow them and their motives.',
 '',
 'This is a good movie if you are looking for a bit of satire, a creepy game and a wonderfully executed female lead. You will get what you sign up for and maybe even a little more.']

In [88]:
calificacion =  click_function(By.XPATH, '//*[@id="__next"]/main/div/section[1]/section/div[3]/section/section/div[3]/div[2]/div[2]/div[1]/div/div[1]/a/span/div/div[2]/div[1]/span[1]').text  

director = click_function(By.XPATH, '//*[@id="__next"]/main/div/section[1]/section/div[3]/section/section/div[3]/div[2]/div[2]/div[2]/ul/li[1]/div/ul/li/a').text

#     # sinopsis = click_function(By.XPATH, '//*[@id="__next"]/main/div/section[1]/section/div[3]/section/section/div[3]/div[2]/div[1]/section/p/span[3]/span/span').text  
# sinopsis = click_function(By.XPATH, '//*[@id="__next"]/main/div/section[1]/section/div[3]/section/section/div[3]/div[2]/div[1]/section/p').text
    

In [136]:
lista_pel[0]

{'Titulo': 'Noche de bodas',
 'Género': 'Action',
 'Año': '2019',
 'Duración': '1h 35min',
 'Calificación': '6,9',
 'Director': 'Matt Bettinelli-Olpin',
 'Protagonistas': ['', '', ''],
 'Sinopsis': 'La noche de bodas de una novia da un giro siniestro cuando sus nuevos suegros la obligan a formar parte de un juego aterrador.',
 'Reseñas': [['6',
   '/10',
   'A decently entertaining dark comedy horror',
   "Ready or Not is a perfectly fine film with a fun concept, I just don't think it takes it far enough!",
   '',
   'I had a lot of fun with the set up and the characters, but felt like there were a few missed opportunities. At times the story also seemed to lag a little bit and surrendered to cliche on a few too many occasions.',
   '',
   'Samara Weaving was a perfect scream queen and an easy character to root for and get on board with.',
   '',
   'Stylistically I liked the film, with its balance of horror, gore, and dark comedy. Again, it just could have been pushed a little further

### Export data

In [137]:
import json


with open("movie.json", "w", encoding="utf-8") as f:
    json.dump(lista_pel, f, ensure_ascii=False, indent=2)

### Import data

In [ ]:
import json

with open("movie.json", "r", encoding="utf-8") as f:
    data = json.load(f)

### Process reviews

In [ ]:
def order_reviews(title, review):
    rating  = -1
    try:
        
        rating = float(review[0])
        rv = review[3:]
    except ValueError:
        print("Error")
        rv = review[1:]
     
    text = " ".join(rv)
    text = "\n".join(rv)
    text = text.replace("\n", " ")

    return {
        "title": title, 
        "rating": rating,
        "Reseña": text
        
    }

In [ ]:
review_list = []
for movie in data:
    for review in movie["Reseñas"]:
        rev = order_reviews(movie["Titulo"], review)
        review_list.append(rev)